# LTSKit CapCut Cloud TTS
Chuyển một SRT thành MP3 trên Colab/Kaggle. Hỗ trợ checkpoint và 1-20 profile song song.

**Cảnh báo:** Đây là API CapCut không chính thức. Nhiều profile vẫn dùng chung IP cloud và có thể tăng SHARK/rate-limit.

In [ ]:
import hashlib, json, math, os, pathlib, re, unicodedata
AUDIO_PROCESSING_VERSION = 1

def sanitize_text(text):
    return ' '.join(''.join(c for c in text if not unicodedata.category(c).startswith(('S', 'C'))).split())
def parse_time(value):
    match = re.fullmatch(r'(\d+):(\d{2}):(\d{2})[,.](\d{3})', value.strip())
    if not match: raise ValueError('Invalid SRT timestamp: ' + value)
    h, m, s, ms = map(int, match.groups())
    return h * 3600 + m * 60 + s + ms / 1000
def parse_srt(text):
    cues = []
    for block in re.split(r'\r?\n\s*\r?\n+', text.lstrip('\ufeff').strip()):
        lines = block.splitlines(); timing = next((i for i, line in enumerate(lines) if '-->' in line), None)
        if timing is None: continue
        left, right = lines[timing].split('-->', 1); start, end = parse_time(left), parse_time(right.split()[0])
        if end <= start: raise ValueError('SRT cue must end after start')
        value = sanitize_text(' '.join(lines[timing + 1:]).replace('\\N', ' '))
        if value: cues.append({'cue_index': len(cues), 'start': start, 'end': end, 'text': value})
    if not cues: raise ValueError('SRT contains no usable cues')
    return cues
def validate_profile_count(value):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 20: raise ValueError('PROFILE_COUNT must be an integer from 1 to 20')
    return value
def cue_fingerprint(cue, voice_id, speed):
    payload = {'text': cue['text'], 'start': cue['start'], 'end': cue['end'], 'voice_id': voice_id, 'speed': speed, 'audio_processing_version': AUDIO_PROCESSING_VERSION}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
def classify_error(message):
    value = message.lower()
    if 'shark' in value or 'risk control' in value or 'device blocked' in value or ('403' in value and 'block' in value): return 'shark'
    if 'system busy' in value or ('1014' in value and ('busy' in value or 'no task' in value)): return 'busy'
    if any(x in value for x in ('connection reset', 'connection aborted', 'connectionerror', 'timed out', 'connecttimeout', 'max retries exceeded')): return 'network'
    return 'api'
def find_audio_url(value, depth=0):
    if depth > 12 or value is None: return None
    if isinstance(value, str):
        if value.startswith('http') and any(x in value.lower() for x in ('audio','speech','tos-','capcut','tiktokcdn','.mp3','.m4a','.wav')): return value
        if value[:1] in ('{','['):
            try: return find_audio_url(json.loads(value), depth + 1)
            except Exception: return None
        return None
    if isinstance(value, dict):
        for key in ('speech_url','audio_url','play_url','download_url','url','audio','speaker_url'):
            found = find_audio_url(value.get(key), depth + 1)
            if found: return found
        for item in value.values():
            found = find_audio_url(item, depth + 1)
            if found: return found
    if isinstance(value, list):
        for item in value:
            found = find_audio_url(item, depth + 1)
            if found: return found
    return None
def atempo_filter(rate):
    remaining, filters = rate, []
    while remaining > 2.000001: filters.append('atempo=2.0'); remaining /= 2
    while remaining < 0.499999: filters.append('atempo=0.5'); remaining /= 0.5
    filters.append('atempo=%.4f' % max(0.5, min(2, remaining)))
    return ','.join(filters)
def audio_plan(source_duration, speed, slot, min_slot=0.15):
    if not math.isfinite(source_duration) or source_duration <= 0: raise ValueError('Cannot measure CapCut audio duration')
    speed = speed if math.isfinite(speed) and speed > 0 else 1.0; usable = max(slot, min_slot)
    limit = usable if source_duration / speed > usable + 0.02 else None
    tempo = source_duration / limit if limit is not None else speed
    return {'tempo': tempo, 'output_limit': limit, 'filter': atempo_filter(tempo)}
def mix_plan(clips, total_sec, work_dir, final_output, batch_size=100):
    if not clips: raise ValueError('No TTS clips to mix')
    batches = []
    clips = sorted(clips, key=lambda c: c['cue_index'])
    for offset in range(0, len(clips), batch_size):
        part = clips[offset:offset + batch_size]; output = os.path.join(work_dir, 'mix-batch-%03d.wav' % len(batches))
        delayed = ['[%d:a]adelay=%d|%d[c%d]' % (i, round(c['start'] * 1000), round(c['start'] * 1000), i) for i, c in enumerate(part)]
        labels = ''.join('[c%d]' % i for i in range(len(part))); silence = len(part)
        filt = ';'.join(delayed + ['[%d:a]%samix=inputs=%d:duration=first:dropout_transition=0:normalize=0[out]' % (silence, labels, len(part) + 1)])
        args = ['ffmpeg', '-y'] + sum((['-i', c['path']] for c in part), []) + ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=mono:d=%s' % total_sec, '-filter_complex', filt, '-map', '[out]', '-t', str(total_sec), output]
        batches.append({'output': output, 'filter': filt, 'args': args})
    inputs = [b['output'] for b in batches]; filt = ''.join('[%d:a]' % i for i in range(len(inputs))) + 'amix=inputs=%d:duration=longest:dropout_transition=0:normalize=0[out]' % len(inputs)
    args = ['ffmpeg', '-y'] + sum((['-i', p] for p in inputs), []) + ['-filter_complex', filt, '-map', '[out]', '-t', str(total_sec), '-codec:a', 'libmp3lame', '-b:a', '128k', final_output]
    return {'batches': batches, 'final_filter': filt, 'final_args': args}


## 1. Cài môi trường
Bật Internet cho Kaggle. Dependency được pin theo commit để tránh thay đổi ngoài ý muốn.

In [ ]:
import shutil, subprocess, sys
SDK_SHA = 'e06da1f4e0c0010354f4e7702f02c18cbdd419a2'
if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True); subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'requests', 'git+https://github.com/K07VN/capcut-tts-api.git@' + SDK_SHA], check=True)
for command, needle in [(['ffmpeg', '-filters'], 'atempo'), (['ffmpeg', '-filters'], 'adelay'), (['ffmpeg', '-filters'], 'amix'), (['ffmpeg', '-encoders'], 'libmp3lame')]:
    result = subprocess.run(command, capture_output=True, text=True); assert result.returncode == 0 and needle in result.stdout, 'FFmpeg missing ' + needle
assert shutil.which('ffprobe'), 'ffprobe is required'
print('Environment ready')


## 2. Cấu hình
Kaggle: dùng `/kaggle/input` cho SRT và `/kaggle/working` cho output/work. Colab: đổi sang `/content` hoặc Google Drive.

In [ ]:
SRT_PATH = '/kaggle/input/subtitles/video.srt'
OUTPUT_PATH = '/kaggle/working/video.mp3'
WORK_DIR = '/kaggle/working/capcut-tts-work'
VOICE_ID = 'BV421_vivn_streaming'
SPEED = 1.0
PROFILE_COUNT = 1  # 1..20; tất cả profile vẫn dùng chung IP cloud
VOICES = {
 'BV421_vivn_streaming': ('Nhỏ Ngọt Ngào', '7252594014782755330'), 'vi_female_huong': ('Giọng Nữ Phổ Thông', '7264854897953083905'),
 'BV074_streaming_dsp': ('Giọng Bé', '7550087831092251920'), 'BV074_streaming': ('Cô Gái Hoạt Ngôn', '7102355709945188865'),
 'vi-VN-HoaiMyNeural': ('Hoai My', '7371666434650280464'), 'vi-VN-NamMinhNeural': ('Nam Minh', '7371666524727153168'),
 'BV075_streaming_vibrato_dsp': ('Việt Méo', '7569450639810465040'), 'BV562_streaming': ('Mai', '7483736254694035984'),
 'multi_female_yangguangnv_uranus_bigtts': ('Ban Mai', '7637456432522218773'), 'multi_female_richgirl_uranus_bigtts': ('Review Phim new', '7637460351541447956'),
 'multi_female_quanweinv_uranus_bigtts': ('Bản Tin 1', '7637458743197732117'), 'multi_female_stokie_uranus_bigtts': ('Review Phim 4', '7637456729696996628'),
 'multi_female_sisi_uranus_bigtts': ('Bản Tin nữ', '7637455857285860629'), 'multi_female_daqi_uranus_bigtts': ('Review Phim 3', '7637451983389019409'),
 'multi_female_xyf04auto_uranus_bigtts': ('Review Phim 2', '7637458743197732117'), 'multi_female_kiwi_uranus_bigtts': ('Sunny Idol', '7637457995882089749'),
 'BV075_streaming_demon_dsp': ('Kenny Đại Đế', '7569442422665661712')
}
print('Vietnamese voices:'); [print(' ', key, '-', value[0]) for key, value in VOICES.items()]


In [ ]:
import concurrent.futures, random, threading, time
from pathlib import Path
import requests
from capcut_tts_api import CapCutClient
RETRY_DELAYS = (0.75, 1.5, 3.0, 6.0)
def random_digits(n):
    value = ''.join(str(random.randint(0, 9)) for _ in range(n)); return value if value[0] != '0' else '7' + value[1:]
def random_device():
    did = random_digits(19); return {'aid':'359289','app_name':'CapCut','appvr':'8.7.0','version_name':'8.7.0','version_code':'8.7.0','channel':'capcutpc_google','device_platform':'mac','device_type':'MacBookPro17,4','device_brand':'MacBookPro17,4','os_version':'15.7.4','device_id':did,'iid':random_digits(19),'tdid':did,'region':'VN','loc':'VN','lan':'vi-VN','pf':'3'}
def atomic_json(path, data):
    temporary = str(path) + '.tmp'; Path(temporary).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8'); os.replace(temporary, path)
def ensure_device(path):
    try:
        value = json.loads(Path(path).read_text(encoding='utf-8'))
        if value.get('device_id') and value.get('iid') and value.get('tdid'): return
    except Exception: pass
    atomic_json(path, random_device())
class ProfileSession:
    def __init__(self, index, root, voice_id, resource_id):
        self.index=index; self.path=Path(root)/'devices'/('profile-%02d.json'%index); self.path.parent.mkdir(parents=True,exist_ok=True); ensure_device(self.path); self.voice_id=voice_id; self.resource_id=resource_id; self.client=None
    def reset(self, rotate=False):
        if rotate: atomic_json(self.path, random_device())
        self.client=None
    def get_client(self):
        if self.client is None: self.client=CapCutClient(device=str(self.path))
        return self.client
    def synth(self, text, output):
        client=self.get_client(); created=client.create_tts_task(texts=text,voice=self.voice_id,resource_id=self.resource_id,rate='1.0'); ret=str((created or {}).get('ret') or '')
        if ret and ret not in ('0','200'): raise RuntimeError('CapCut API ret='+ret+(' system busy' if ret=='1014' else ''))
        tasks=((created.get('data')or{}).get('tasks')or[])
        if not tasks: raise RuntimeError('CapCut returned no task')
        url=find_audio_url(created); started=time.time()
        while not url and time.time()-started<90:
            queried=client.query_tts_task(tasks[0]['id'],tasks[0]['token']); qtasks=((queried.get('data')or{}).get('tasks')or[]); status=str(qtasks[0].get('status','')).lower() if qtasks else ''
            if status in ('failed','fail','error'): raise RuntimeError('CapCut TTS task failed')
            if status in ('success','succeed','succeeded','done','completed','finish','finished'): url=find_audio_url(queried) or find_audio_url(qtasks[0].get('payload'))
            if not url: time.sleep(1)
        if not url: raise RuntimeError('CapCut TTS timed out')
        temporary=str(output)+'.part'; size=0
        with requests.get(url,stream=True,timeout=60) as response:
            response.raise_for_status()
            with open(temporary,'wb') as file:
                for chunk in response.iter_content(65536):
                    if chunk: file.write(chunk); size += len(chunk)
        if size < 64: raise RuntimeError('Downloaded audio is too small')
        os.replace(temporary,output)
def probe_duration(path):
    result=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(path)],capture_output=True,text=True,check=True); return float(result.stdout.strip())
def fit_audio(source, output, speed, slot):
    plan=audio_plan(probe_duration(source),speed,slot); args=['ffmpeg','-y','-i',str(source),'-ac','1','-ar','48000','-filter:a',plan['filter']]
    if plan['output_limit'] is not None: args += ['-t',str(plan['output_limit'])]
    subprocess.run(args+[str(output)],capture_output=True,check=True); duration=probe_duration(output)
    if Path(output).stat().st_size<=44: raise RuntimeError('Fitted WAV is invalid')
    return duration


In [ ]:
def run_capcut_tts():
    started=time.time(); profiles=validate_profile_count(PROFILE_COUNT)
    if VOICE_ID not in VOICES: raise ValueError('Unknown VOICE_ID: '+VOICE_ID)
    if not math.isfinite(SPEED) or SPEED<=0: raise ValueError('SPEED must be positive')
    srt=Path(SRT_PATH); output=Path(OUTPUT_PATH); root=Path(WORK_DIR); clips_dir=root/'clips'; source_dir=root/'source'; root.mkdir(parents=True,exist_ok=True); clips_dir.mkdir(exist_ok=True); source_dir.mkdir(exist_ok=True); output.parent.mkdir(parents=True,exist_ok=True)
    cues=parse_srt(srt.read_text(encoding='utf-8-sig')); manifest_path=root/'manifest.json'
    try: manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
    except Exception: manifest={'version':1,'entries':[]}
    expected={c['cue_index']:cue_fingerprint(c,VOICE_ID,SPEED) for c in cues}; complete={}
    for entry in manifest.get('entries',[]):
        path=root/entry.get('path','')
        if expected.get(entry.get('cue_index'))==entry.get('fingerprint') and path.is_file() and path.stat().st_size>44: complete[entry['cue_index']]=entry
    sessions=[ProfileSession(i+1,root,VOICE_ID,VOICES[VOICE_ID][1]) for i in range(profiles)]; lock=threading.Lock(); stop=threading.Event(); metrics={'completed':len(complete),'sharks':0,'busy_retries':0,'network_retries':0,'sharks_by_profile':[0]*profiles}
    pending=[c for c in cues if c['cue_index'] not in complete]; next_cue=0
    def process(cue, session):
        if stop.is_set(): return
        source=source_dir/('%06d.mp3'%cue['cue_index']); fitted=clips_dir/('%06d.wav'%cue['cue_index'])
        last=None
        for attempt in range(5):
            try:
                session.synth(cue['text'],source); duration=fit_audio(source,fitted,SPEED,cue['end']-cue['start']); entry={'cue_index':cue['cue_index'],'fingerprint':expected[cue['cue_index']],'path':str(fitted.relative_to(root)),'start':cue['start'],'dur':duration}
                with lock:
                    complete[cue['cue_index']]=entry; metrics['completed']+=1; atomic_json(manifest_path,{'version':1,'entries':[complete[i] for i in sorted(complete)]}); print('Cue %d/%d complete (profile %d)'%(metrics['completed'],len(cues),session.index))
                return
            except Exception as error:
                last=error; kind=classify_error(str(error))
                with lock:
                    if kind=='shark': metrics['sharks']+=1; metrics['sharks_by_profile'][session.index-1]+=1
                    elif kind=='busy': metrics['busy_retries']+=1
                    elif kind=='network': metrics['network_retries']+=1
                if kind=='api' or attempt==4: break
                session.reset(rotate=kind=='shark') if kind in ('shark','network') else None; time.sleep(RETRY_DELAYS[attempt])
        stop.set(); raise RuntimeError('Cue %d failed after retries: %s'%(cue['cue_index']+1,last))
    def worker(session):
        nonlocal next_cue
        while not stop.is_set():
            with lock:
                if next_cue >= len(pending): return
                cue = pending[next_cue]; next_cue += 1
            process(cue, session)
    errors=[]
    with concurrent.futures.ThreadPoolExecutor(max_workers=profiles) as executor:
        futures=[executor.submit(worker, session) for session in sessions[:min(profiles,len(pending))]]
        for future in concurrent.futures.as_completed(futures):
            try: future.result()
            except Exception as error: errors.append(str(error)); stop.set()
    if errors: raise RuntimeError(errors[0]+'; checkpoint kept at '+str(manifest_path))
    clips=[{'cue_index':e['cue_index'],'path':str(root/e['path']),'start':e['start'],'dur':e['dur']} for e in complete.values()]; total=max(max(c['end'] for c in cues),max(c['start']+c['dur'] for c in clips),0.5); private=output.with_suffix(output.suffix+'.part.mp3'); plan=mix_plan(clips,total,str(root),str(private))
    for batch in plan['batches']: subprocess.run(batch['args'],capture_output=True,check=True)
    subprocess.run(plan['final_args'],capture_output=True,check=True); assert probe_duration(private)>0; os.replace(private,output)
    print(json.dumps({**metrics,'profiles':profiles,'cues':len(cues),'elapsed_sec':round(time.time()-started,2),'checkpoint':str(manifest_path),'output':str(output)},ensure_ascii=False,indent=2)); return str(output)
RESULT = run_capcut_tts()
